# Gauge Audit Map

Validates which USGS gauges in Colorado are returning live data, and cross-checks
the configured `ACTIVE_GAUGES` against the latest audit snapshot.

**Inputs**: latest GeoJSON in `../gauge_audit_logs/` (produced by
`pipeline/scripts/audit_gauges.py`).

**Sections**
1. Load latest audit snapshot
2. Rating distribution across all CO stream gauges
3. Filter to gauges with active data
4. Top-rated gauges (sanity-check the data)
5. Cross-reference with the configured `ACTIVE_GAUGES`
6. Interactive Folium map with parameter filter


## Setup

In [1]:
import glob
import os
import sys
from pathlib import Path

import folium
import geopandas as gpd
import pandas as pd
from branca.element import MacroElement
from jinja2 import Template

REPO_ROOT = Path("..").resolve()
AUDIT_DIR = REPO_ROOT / "gauge_audit_logs"

PARAMS = {
    "discharge":     {"label": "Discharge",     "color": "#2563eb"},
    "gage_height":   {"label": "Gage Height",   "color": "#16a34a"},
    "water_temp":    {"label": "Water Temp",    "color": "#d97706"},
    "precipitation": {"label": "Precipitation", "color": "#7c3aed"},
}

# Make `pipeline.config.rivers` importable from the repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


## 1. Load latest audit snapshot

In [2]:
latest = max(
    glob.glob(str(AUDIT_DIR / "gauge_audit_*.geojson")),
    key=os.path.getmtime,
)
print(f"Loading: {latest}")

gdf = gpd.read_file(latest)
print(f"Total gauges in snapshot: {len(gdf)}")
gdf.head(3)


Loading: /home/bryang/Dev_Space/ml_projects/riffle/gauge_audit_logs/gauge_audit_20260410_163855.geojson
Total gauges in snapshot: 6104


,gauge_id,name,site_type,huc_code,county,altitude_ft,drainage_area_sqmi,contrib_drainage_area_sqmi,discharge_value,discharge_unit,...,precipitation_value,precipitation_unit,precipitation_last_at,precipitation_hours_ago,precipitation_active,reading_rating,reading_rating_4,last_any_reading_at,parameter_count,geometry
0,06611000,"COLORADO CREEK NEAR SPICER, CO.",ST,101800010102,Jackson County,8381.0,25.8,NaN,None,None,...,None,None,NaT,NaN,False,0,0,NaT,0,POINT (-106.50198 40.44193)
1,06611100,"GRIZZLY CREEK NEAR SPICER, CO.",ST,101800010106,Jackson County,8234.0,118.0,NaN,None,None,...,None,None,NaT,NaN,False,0,0,NaT,0,POINT (-106.44975 40.49331)
2,06611200,"BUFFALO CREEK NEAR HEBRON, CO.",ST,101800010105,Jackson County,8190.0,56.3,NaN,None,None,...,None,None,NaT,NaN,False,0,0,NaT,0,POINT (-106.3692 40.52304)


## 2. Rating distribution

`reading_rating_4` counts how many of the 4 monitored parameters
(discharge, gage height, water temp, precipitation) reported a reading
within the last 48 hours.

In [3]:
rating_counts = (
    gdf["reading_rating_4"]
    .value_counts()
    .sort_index(ascending=False)
    .rename("gauges")
    .to_frame()
)
rating_counts["pct"] = (rating_counts["gauges"] / len(gdf) * 100).round(1)
rating_counts


,gauges,pct
reading_rating_4,,
4,11,0.2
3,74,1.2
2,229,3.8
1,32,0.5
0,5758,94.3


## 3. Filter to gauges with at least one active parameter

In [4]:
before = len(gdf)
gdf = gdf[gdf["reading_rating_4"] > 0].copy()
print(f"Filtered: {before} → {len(gdf)} gauges with at least 1 active parameter")


Filtered: 6104 → 346 gauges with at least 1 active parameter


## 4. Top-rated gauges

Sanity-check the audit data without opening the map. Sorted by rating, then
by `discharge_last_at` so the freshest readings surface first.

In [5]:
preview_cols = [
    "gauge_id", "name", "county",
    "reading_rating_4", "reading_rating",
    "discharge_last_at", "gage_height_last_at",
    "water_temp_last_at", "precipitation_last_at",
]
(
    gdf[preview_cols]
    .sort_values(
        ["reading_rating_4", "discharge_last_at"],
        ascending=[False, False],
    )
    .head(20)
)


,gauge_id,name,county,reading_rating_4,reading_rating,discharge_last_at,gage_height_last_at,water_temp_last_at,precipitation_last_at
838,09058000,"COLORADO RIVER NEAR KREMMLING, CO",Grand County,4,3,2026-04-10 16:15:00,2026-04-10 16:15:00,2026-04-10 16:15:00,2026-04-10 16:15:00
1007,09095500,"COLORADO RIVER NEAR CAMEO, CO.",Mesa County,4,3,2026-04-10 16:15:00,2026-04-10 16:15:00,2026-04-10 16:15:00,2026-04-10 16:15:00
1133,09136100,"NORTH FK GUNNISON RIVER ABOVE MOUTH NR LAZEAR, CO",Delta County,4,3,2026-04-10 16:15:00,2026-04-10 16:15:00,2026-04-10 16:15:00,2026-04-10 16:15:00
1168,09147500,"UNCOMPAHGRE RIVER AT COLONA, CO",Ouray County,4,3,2026-04-10 16:15:00,2026-04-10 16:15:00,2026-04-10 16:15:00,2026-04-10 16:15:00
479,07103970,MONUMENT CR ABV WOODMEN RD AT COLORADO SPRINGS...,El Paso County,4,3,2026-04-10 15:50:00,2026-04-10 15:50:00,2026-04-10 15:50:00,2026-04-10 15:50:00
499,07105800,"FOUNTAIN CREEK AT SECURITY, CO",El Paso County,4,3,2026-04-10 15:50:00,2026-04-10 15:50:00,2026-04-10 15:50:00,2026-04-10 15:50:00
489,07104905,"MONUMENT CREEK AT BIJOU ST. AT COLO. SPRINGS, CO",El Paso County,4,3,2026-04-10 15:45:00,2026-04-10 15:45:00,2026-04-10 15:45:00,2026-04-10 15:45:00
756,09034250,"COLORADO RIVER AT WINDY GAP, NEAR GRANBY, CO.",Grand County,4,3,2026-04-10 15:45:00,2026-04-10 15:45:00,2026-04-10 15:45:00,2026-04-10 15:45:00
602,07126485,"PURGATOIRE RIVER AT ROCK CROSSING NR TIMPAS, CO.",Las Animas County,4,3,2026-04-10 15:40:00,2026-04-10 15:40:00,2026-04-10 15:30:00,2026-04-10 15:40:00
783,09041090,MUDDY CREEK ABOVE ANTELOPE CREEK NR. KREMMLING...,Grand County,4,3,2026-04-10 15:30:00,2026-04-10 16:30:00,2026-04-10 16:30:00,2026-04-10 16:30:00


## 5. Cross-reference with configured `ACTIVE_GAUGES`

The point of the audit. Loads `GAUGES` from `pipeline/config/rivers.py`,
left-joins against the audit snapshot, and flags any configured gauge whose
`reading_rating < 2` (missing one or more of discharge / gage height / water temp)
or that's missing from the audit entirely.

In [6]:
from pipeline.config.rivers import GAUGES

configured = pd.DataFrame(GAUGES)
configured["usgs_gauge_id"] = configured["usgs_gauge_id"].astype(str)

audit_subset = gdf[[
    "gauge_id", "reading_rating", "reading_rating_4",
    "discharge_last_at", "gage_height_last_at",
    "water_temp_last_at", "precipitation_last_at",
]].copy()
audit_subset["gauge_id"] = audit_subset["gauge_id"].astype(str)

merged = configured.merge(
    audit_subset,
    left_on="usgs_gauge_id",
    right_on="gauge_id",
    how="left",
)

def status(row):
    if pd.isna(row["reading_rating"]):
        return "MISSING from audit"
    if row["reading_rating"] < 2:
        return f"STALE (rating {int(row['reading_rating'])}/3)"
    return f"OK ({int(row['reading_rating'])}/3)"

merged["status"] = merged.apply(status, axis=1)

merged[[
    "usgs_gauge_id", "name", "river", "active", "status",
    "discharge_last_at", "gage_height_last_at", "water_temp_last_at",
]]


,usgs_gauge_id,name,river,active,status,discharge_last_at,gage_height_last_at,water_temp_last_at
0,09035800,Spinney Reservoir,South Platte,False,MISSING from audit,NaT,NaT,NaT
1,09036000,Eleven Mile Canyon,South Platte,False,OK (2/3),2026-04-10 16:25:00,2026-04-10 16:25:00,NaT
2,09033300,Deckers,South Platte,False,MISSING from audit,NaT,NaT,NaT
3,09034500,Cheesman Canyon,South Platte,False,MISSING from audit,NaT,NaT,NaT
4,07091200,Salida,Arkansas River,True,OK (2/3),2026-04-10 15:45:00,2026-04-10 15:45:00,NaT
5,07096000,Cañon City,Arkansas River,False,STALE (rating 1/3),2006-10-01 05:30:00,NaT,2026-04-10 16:15:00
6,09081600,Ruedi to Basalt,Fryingpan River,False,OK (2/3),2026-04-10 16:00:00,2026-04-10 16:00:00,NaT
7,09085000,Glenwood Springs,Roaring Fork River,True,OK (3/3),2026-04-10 15:45:00,2026-04-10 15:45:00,2026-04-10 15:45:00
8,09057500,Silverthorne,Blue River,True,OK (2/3),2026-04-10 16:00:00,2026-04-10 16:00:00,NaT
9,06752000,Canyon Mouth,Cache la Poudre,False,MISSING from audit,NaT,NaT,NaT


### Configured gauges that need attention

In [7]:
needs_attention = merged[merged["status"] != merged["status"].where(
    merged["status"].str.startswith("OK")
).fillna("")]
needs_attention = merged[~merged["status"].str.startswith("OK")]
needs_attention[["usgs_gauge_id", "name", "river", "active", "status"]]


,usgs_gauge_id,name,river,active,status
0,09035800,Spinney Reservoir,South Platte,False,MISSING from audit
2,09033300,Deckers,South Platte,False,MISSING from audit
3,09034500,Cheesman Canyon,South Platte,False,MISSING from audit
5,07096000,Cañon City,Arkansas River,False,STALE (rating 1/3)
9,06752000,Canyon Mouth,Cache la Poudre,False,MISSING from audit


### Currently active in config

The 6 gauges that survive the audit and are still wired into the pipeline.
This is the working set the ML flow ingests against.

In [8]:
ACTIVE_IDS = {g["usgs_gauge_id"] for g in GAUGES if g["active"]}
print(f"{len(ACTIVE_IDS)} active gauges in config")

active_view = (
    merged[merged["usgs_gauge_id"].isin(ACTIVE_IDS)]
    [[
        "usgs_gauge_id", "name", "river", "status",
        "discharge_last_at", "gage_height_last_at", "water_temp_last_at",
    ]]
    .reset_index(drop=True)
)
active_view


23 active gauges in config


,usgs_gauge_id,name,river,status,discharge_last_at,gage_height_last_at,water_temp_last_at
0,07091200,Salida,Arkansas River,OK (2/3),2026-04-10 15:45:00,2026-04-10 15:45:00,NaT
1,09085000,Glenwood Springs,Roaring Fork River,OK (3/3),2026-04-10 15:45:00,2026-04-10 15:45:00,2026-04-10 15:45:00
2,09057500,Silverthorne,Blue River,OK (2/3),2026-04-10 16:00:00,2026-04-10 16:00:00,NaT
3,09070000,Glenwood Springs,Colorado River,OK (2/3),2026-04-10 16:15:00,2026-04-10 16:15:00,NaT
4,06700000,Above Cheesman Lake,South Platte,OK (2/3),2026-04-10 15:45:00,2026-04-10 15:45:00,NaT
5,06701900,Trumbull (Brush Creek),South Platte,OK (2/3),2026-04-10 15:45:00,2026-04-10 15:45:00,NaT
6,06716500,Lawson,Clear Creek,OK (2/3),2026-04-10 15:45:00,2026-04-10 15:45:00,NaT
7,09034250,Windy Gap,Colorado River,OK (3/3),2026-04-10 15:45:00,2026-04-10 15:45:00,2026-04-10 15:45:00
8,09046600,Dillon,Blue River,OK (3/3),2026-04-10 16:00:00,2026-04-10 16:00:00,2026-04-10 16:00:00
9,09058000,Kremmling,Colorado River,OK (3/3),2026-04-10 16:15:00,2026-04-10 16:15:00,2026-04-10 16:15:00


## 6. Interactive map

Color-coded by `reading_rating_4`. Use the parameter filter (bottom left) to
narrow to gauges that report a specific combination of measurements.

### Tooltip helpers

In [9]:
def _val(row, label):
    v = row.get(f"{label}_value")
    u = row.get(f"{label}_unit") or ""
    t = row.get(f"{label}_last_at") or "unknown"
    active = row.get(f"{label}_active", False)
    dot = "&#x1F7E2;" if active else "&#x1F534;"
    if v is None:
        return f"{dot} N/A"
    return f"{dot} {v} {u} &nbsp;<span style='color:#888'>@ {t}</span>"


def make_tooltip(row):
    rating = int(row.get("reading_rating_4", 0))
    stars = "&#9733;" * rating + "&#9734;" * (4 - rating)
    return f"""
<div style="font-family: sans-serif; font-size: 12px; min-width: 300px; padding: 2px 4px;">
  <div style="font-size: 14px; font-weight: bold; margin-bottom: 2px;">{row['gauge_id']}</div>
  <div style="color: #444; margin-bottom: 6px;">{row['name']}</div>
  <table style="width: 100%; border-collapse: collapse; line-height: 1.6;">
    <tr><td style="color:#555; padding-right:8px;">Discharge</td><td>{_val(row, 'discharge')}</td></tr>
    <tr><td style="color:#555; padding-right:8px;">Gage Height</td><td>{_val(row, 'gage_height')}</td></tr>
    <tr><td style="color:#555; padding-right:8px;">Water Temp</td><td>{_val(row, 'water_temp')}</td></tr>
    <tr><td style="color:#555; padding-right:8px;">Precipitation</td><td>{_val(row, 'precipitation')}</td></tr>
  </table>
  <div style="margin-top: 6px; font-size: 13px;"><b>Rating:</b> {stars} &nbsp;{rating}/4</div>
</div>
"""


### Marker data prep

In [10]:
marker_data = []
for _, row in gdf.iterrows():
    r = row.to_dict()
    active_params = [p for p in PARAMS if r.get(f"{p}_active")]
    marker_data.append({
        "lat": row.geometry.y,
        "lon": row.geometry.x,
        "params": active_params,
    })

print(f"Prepared {len(marker_data)} markers")
marker_data[:3]


Prepared 346 markers


[{'lat': 40.496111,
  'lon': -105.864444,
  'params': ['discharge', 'gage_height']},
 {'lat': 40.936638888888886,
  'lon': -106.33919444444444,
  'params': ['discharge', 'gage_height']},
 {'lat': 39.162778,
  'lon': -105.309722,
  'params': ['discharge', 'gage_height', 'precipitation']}]

### Build base map and add markers

In [11]:
m = folium.Map(location=[39.0, -105.5], zoom_start=7)
marker_group = folium.FeatureGroup(name="All CO gauges").add_to(m)

for _, row in gdf.iterrows():
    r = row.to_dict()
    rating = int(r.get("reading_rating_4", 0))
    color = {0: "#9ca3af", 1: "#eab308", 2: "#f97316", 3: "#3b82f6", 4: "#16a34a"}[rating]
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        tooltip=folium.Tooltip(make_tooltip(r)),
    ).add_to(marker_group)

# Gold-ring overlay for gauges in our active config
active_group = folium.FeatureGroup(name="Active in config", show=True).add_to(m)
active_in_view = gdf[gdf["gauge_id"].astype(str).isin(ACTIVE_IDS)]
for _, row in active_in_view.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=11,
        color="#fbbf24",
        weight=3,
        fill=False,
        opacity=1.0,
    ).add_to(active_group)

folium.LayerControl(collapsed=False).add_to(m)

print(f"Added {len(gdf)} markers, highlighted {len(active_in_view)} active config gauges")


Added 346 markers, highlighted 23 active config gauges


### Inject parameter filter panel

In [12]:
marker_data_js = (
    str(marker_data)
    .replace("'", '"')
    .replace("True", "true")
    .replace("False", "false")
)
map_var = m.get_name()
marker_group_var = marker_group.get_name()

filter_js = MacroElement()
filter_js._template = Template(f"""
{{% macro script(this, kwargs) %}}

var markerData = {marker_data_js};
var markerLayers = [];

// Only walk the main marker_group so the active-config overlay doesn't get
// included (its markers don't correspond to entries in markerData).
{marker_group_var}.eachLayer(function(sublayer) {{
    if (sublayer.getLatLng) {{
        markerLayers.push(sublayer);
    }}
}});

function applyFilter() {{
    var checked = [];
    document.querySelectorAll('.param-checkbox:checked').forEach(function(cb) {{
        checked.push(cb.value);
    }});
    for (var i = 0; i < markerLayers.length; i++) {{
        var params = markerData[i].params;
        var show = checked.every(function(p) {{ return params.indexOf(p) !== -1; }});
        if (show) {{
            markerLayers[i].setStyle({{opacity: 1, fillOpacity: 0.75}});
            if (markerLayers[i]._path) markerLayers[i]._path.style.pointerEvents = '';
        }} else {{
            markerLayers[i].setStyle({{opacity: 0, fillOpacity: 0}});
            if (markerLayers[i]._path) markerLayers[i]._path.style.pointerEvents = 'none';
        }}
    }}
}}

var panel = L.control({{position: 'bottomleft'}});
panel.onAdd = function(map) {{
    var div = L.DomUtil.create('div', 'param-filter-panel');
    div.style.cssText = 'background:white; padding:10px 14px; border-radius:6px; box-shadow:0 2px 6px rgba(0,0,0,0.2); font-family:sans-serif; font-size:13px;';
    div.innerHTML = '<div style="font-weight:bold; margin-bottom:6px;">Filter by parameter</div>'
        + '<div style="color:#888; font-size:11px; margin-bottom:8px;">Show gauges with ALL selected</div>'
        + '<label style="display:block; margin-bottom:4px;"><input type="checkbox" class="param-checkbox" value="discharge" checked> Discharge</label>'
        + '<label style="display:block; margin-bottom:4px;"><input type="checkbox" class="param-checkbox" value="gage_height" checked> Gage Height</label>'
        + '<label style="display:block; margin-bottom:4px;"><input type="checkbox" class="param-checkbox" value="water_temp" checked> Water Temp</label>'
        + '<label style="display:block; margin-bottom:4px;"><input type="checkbox" class="param-checkbox" value="precipitation"> Precipitation</label>';
    L.DomEvent.disableClickPropagation(div);
    return div;
}};
panel.addTo({map_var});

setTimeout(function() {{
    document.querySelectorAll('.param-checkbox').forEach(function(cb) {{
        cb.addEventListener('change', applyFilter);
    }});
    applyFilter();
}}, 300);

{{% endmacro %}}
""")

_ = m.get_root().add_child(filter_js)


### Render

In [13]:
m